## Lab07: RAG (Retrieval-Augmented-Generation)

RAG (Retrieval-Augmented Generation) is a powerful framework in Natural Language Processing (NLP) that enhances the performance of language models by combining traditional generative models with external knowledge retrieval. This hybrid approach allows models to retrieve relevant information from a large corpus (like a database or document collection) and incorporate this information into the generation process. It is particularly useful when a model needs to answer questions, generate content, or provide explanations based on real-time or domain-specific data.



In [1]:
# In order to load models, we need first to choose the device and perform the login to Huggingface.

import torch

# Detect the device available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", device) # You may need to execute "pip install torch --index-url https://download.pytorch.org/whl/cu124" (or other versions in place of the 12.4) in order to use the GPU

# HuggingFace login: it is needed only once

#----------------Google Colab only---------------------
# from huggingface_hub import login
#from google.colab import userdata
#login(userdata.get('HF_TOKEN'))
#----------------Google Colab only---------------------

#----------------Local Runtime only---------------------
import os
from dotenv import load_dotenv # pip install python-dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HF_TOKEN") # Create this constant in the file .env (put it in the root dir of your project)
login(token=hf_token)
#----------------Local Runtime only---------------------

Using device:  cuda


In [3]:
# These functions are adapted from the previous Lab.
!pip install bitsandbytes
from transformers import BitsAndBytesConfig # pip install bitsandbytes

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
    pipeline,
    TextGenerationPipeline,
)


def load_model(model_id: str, quantization: bool = False) -> TextGenerationPipeline:
    """Loads a pretrained causal language model and builds a configured text-generation pipeline.

    Note: The initial execution may take significant time to download the model weights.

    Args:
        model_id (str): The Hugging Face model identifier or local path.
        quantization (bool, optional): Whether to use quantization. Defaults to False.

    Returns:
        TextGenerationPipeline: A Hugging Face text-generation pipeline initialized with
            the float16 causal LM, its tokenizer, and custom generation configuration.
    """
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    if quantization:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto",
                                                     quantization_config=quantization_config)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto", )
    tokenizer = AutoTokenizer.from_pretrained(model_id, clean_up_tokenization_spaces=False)

    # Configure generation parameters
    gen_config = GenerationConfig.from_model_config(model.config)
    #gen_config.temperature = 0.1              # Regulate randomness, use with do_sample = True
    # gen_config.max_length = None              # Since max_new_tokens is already set
    #gen_config.repetition_penalty = 2.0  # Penalize repetition
    gen_config.no_repeat_ngram_size = 3  # Avoid repeating trigrams (IMPORTANT: do not set this parameter if you target code generation, because code naturally repeats short token sequences)
    gen_config.do_sample = False  # Set to False in order to use Greedy or Beam search
    gen_config.num_beams = 3  # Use with do_sample = False
    gen_config.eos_token_id = tokenizer.eos_token_id  # End generation at EOS token
    gen_config.pad_token_id = tokenizer.pad_token_id
    gen_config.early_stopping = True

    model.generation_config = gen_config

    text_generation_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
    )

    return text_generation_pipeline


## Usage example:
text_generation_pipeline = load_model(model_id="meta-llama/Llama-3.2-3B-Instruct", quantization=True)
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]
outputs = text_generation_pipeline(messages, max_new_tokens = 200)
print(outputs[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


That would be Paris.


In [4]:
!pip install pymupdf
import pymupdf

def extract_text_from_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    text = ""
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        text+=page.get_text()
    return text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 144.1 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 196.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pymupdf]m3/4 [pymupdf]


In [5]:
import os
import glob
from pathlib import Path

# Extract text from all PDF files
pdf_texts = {}
# directory = "."
directory=Path("/kaggle/input/datasets/alainrizzi/llm4se-lab07")
# pdf_files = glob.glob(os.path.join(directory,"paper*.pdf"))
pdf_files = sorted(directory.rglob("paper*.pdf"))
#print("Current directory:", os.getcwd())
#print("Matching files:", pdf_files)
n = 0
for pdf_file in pdf_files:
    pdf_texts[n] = extract_text_from_pdf(pdf_file)
    n = n+1

#Display the text from all the PDF files
for pdf_file, txt in pdf_texts.items():
    print(f"--- {pdf_file} ---")
    print(txt[:500])  # Display the first 500 characters of each document
    print("\n")

--- 0 ---
XIWU: A BASIS FLEXIBLE AND LEARNABLE LLM FOR HIGH
ENERGY PHYSICS
Zhengde Zhang1, Yiyu Zhang1, Haodong Yao1, Jianwen Luo2, Rui Zhao1, Bo Huang1, Jiameng Zhao3,1,
Yipu Liao1, Ke Li1, Lina Zhao1, Jun Cao1, Fazhi Qi1,∗, and Changzheng Yuan1,†
1Institute of High Energy Physics, Chinese Academy of Sciences, Beijing 100049, China
2School of Physical Sciences, University of Chinese Academy of Sciences, Beijing 100049, China
3School of Computer and Artificial Intelligence, ZhengZhou University, Henan 450


--- 1 ---
1
Towards Greener LLMs: Bringing Energy-Efficiency
to the Forefront of LLM Inference
Jovan Stojkovic, Esha Choukse†, Chaojie Zhang†, Inigo Goiri†, Josep Torrellas
University of Illinois at Urbana-Champaign
†Microsoft Azure Research - Systems
Abstract—With the ubiquitous use of modern large language
models (LLMs) across industries, the inference serving for these
models is ever expanding. Given the high compute and memory
requirements of modern LLMs, more and more top-of-th

### Creating an index of vectors to represent the documents

To perform efficient searches, we need to convert our text data into numerical vectors. To do so, we will use the first step of the BERT transformer.

Since our full pdf files are very long to be fed as input into BERT, we perform a step in which we create a structure where we associate a document number to its abstract, and in a separate dictionary we associate a document number to its full text.


In [6]:
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np




#import the Bert pretrained model from the transformers library
model_embeddings = AutoModel.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
### Don't forget to move it to the device
model_embeddings.to(device)
#initialization of the dictionary of abstracts. Substitute this with the abstracts of the 10 papers considered as sources for RAG
#(we could use functions to read the PDFs to "cut" the abstracts from the papers. For simplicity reasons, we will copy and paste them)
abstracts_dict = {
    0: "Abstract of paper 0: Large Language Models (LLMs) are undergoing a period of rapid updates and changes, with stateof-the-art (SOTA) model frequently being replaced. When applying LLMs to a specific scientific field, it’s challenging to acquire unique domain knowledge while keeping the model itself advanced. To address this challenge, a sophisticated large language model system named as Xiwu has been developed, allowing you switch between the most advanced foundation models and quickly teach the model domain knowledge. In this work, we will report on the best practices for applying LLMs in the field of high-energy physics (HEP), including: a seed fission technology is proposed and some data collection and cleaning tools are developed to quickly obtain domain AI-Ready dataset; a just-in-time learning system is implemented based on the vector store technology; an on-the-fly fine-tuning system has been developed to facilitate rapid training under a specified foundation model. The results show that Xiwu can smoothly switch between foundation models such as LLaMA, Vicuna, ChatGLM and Grok-1. The trained Xiwu model is significantly outperformed the benchmark model on the HEP knowledge Q&A and code generation. This strategy significantly enhances the potential for growth of our model’s performance, with the hope of surpassing GPT-4 as it evolves with the development of open-source models. This work provides a customized LLM for the field of HEP, while also offering references for applying LLM to other fields, the corresponding codes are available on Github https://github.comzhang/zhengde0225/Xiwu",
    1: "Abstract of paper 1: With the ubiquitous use of modern large language models (LLMs) across industries, the inference serving for these models is ever expanding. Given the high compute and memory requirements of modern LLMs, more and more top-of-theline GPUs are being deployed to serve these models. Energy availability has come to the forefront as the biggest challenge for data center expansion to serve these models. In this paper, we present the trade-offs brought up by making energy efficiency the primary goal of LLM serving under performance SLOs. We show that depending on the inputs, the model, and the service-level agreements, there are several knobs available to the LLM inference provider to use for being energy efficient. We characterize the impact of these knobs on the latency, throughput, as well as the energy. By exploring these tradeoffs, we offer valuable insights into optimizing energy usage without compromising on performance, thereby paving the way for sustainable and cost-effective LLM deployment in data center environments.",
    2: "Abstract of paper 2: The rapid adoption of large language models (LLMs) has led to significant advances in natural language processing and text generation. However, the energy consumed through LLM model inference remains a major challenge for sustainable AI deployment. To address this problem, we model the workload-dependent energy consumption and runtime of LLM inference tasks on heterogeneous GPU-CPU systems. By conducting an extensive characterization study of several state-of-the-art LLMs and analyzing their energy and runtime behavior across different magnitudes of input prompts and output text, we develop accurate (𝑅2 > 0.96) energy and runtime models for each LLM. We employ these models to explore an offline, energy-optimal LLM workload scheduling framework. Through a case study, we demonstrate the advantages of energy and accuracy aware scheduling compared to existing best practices.",
    3: "Abstract of paper 3: The growing demand for efficient and scalable AI solutions has driven research into optimizing the performance and energy efficiency of computational infrastructures. The novel concept of redesigning inference clusters and modifying the GPT-Neo model offers a significant advancement in addressing the computational and environmental challenges associated with AI deployment. By developing a novel cluster architecture and implementing strategic architectural and algorithmic changes, the research achieved substantial improvements in throughput, latency, and energy consumption. The integration of advanced interconnect technologies, high-bandwidth memory modules, and energy-efficient power management techniques, alongside software optimizations, enabled the redesigned clusters to outperform baseline models significantly. Empirical evaluations demonstrated superior scalability, robustness, and environmental sustainability, emphasizing the potential for more sustainable AI technologies. The findings underscore the importance of balancing performance with energy efficiency and provide a robust framework for future research and development in AI optimization. The research contributes valuable insights into the design and deployment of more efficient and environmentally responsible AI systems.",
    4: "Abstract of paper 4: Establishing building energy models (BEMs) for building design and analysis poses significant challenges due to demanding modeling efforts, expertise to use simulation software, and building science knowledge in practice. These make building modeling labor-intensive, hindering its widespread adoptions in building development. Therefore, to overcome these challenges in building modeling with enhanced automation in modeling practice, this paper proposes Eplus-LLM (EnergyPlus-Large Language Model) as the auto-building modeling platform, building on a fine-tuned large language model (LLM) to directly translate natural language description of buildings to established building models of various geometries, occupancy scenarios, and equipment loads. Through fine-tuning, the LLM (i.e., T5) is customized to digest natural language and simulation demands from users and convert human descriptions into EnergyPlus modeling files. Then, the Eplus-LLM platform realizes the automated building modeling through invoking the API of simulation software (i.e., the EnergyPlus engine) to simulate the auto-generated model files and output simulation results of interest. The validation process, involving four different types of prompts, demonstrates that Eplus-LLM reduces over 95% modeling efforts and achieves 100% accuracy in establishing BEMs while being robust to interference in usage, including but not limited to different tones, misspells, omissions, and redundancies. Overall, this research serves as the pioneering effort to customize LLM for auto-modeling purpose (directly build-up building models from natural language), aiming to provide a user-friendly human-AI interface that significantly reduces building modeling efforts. This work also further facilitates large-scale building model efforts, e.g., urban building energy modeling (UBEM), in modeling practice.",
    5: "Abstract of paper 5: The rapid evolution and widespread adoption of generative large language models (LLMs) have made them a pivotal workload in various applications. Today, LLM inference clusters receive a large number of queries with strict Service Level Objectives (SLOs). To achieve the desired performance, these models execute on power-hungry GPUs causing the inference clusters to consume large amount of energy and, consequently, result in excessive carbon emissions. Fortunately, we find that there is a great opportunity to exploit the heterogeneity in inference compute properties and fluctuations in inference workloads, to significantly improve energy-efficiency. However, such a diverse and dynamic environment creates a large search-space where different system configurations (e.g., number of instances, model parallelism, and GPU frequency) translate into different energy-performance trade-offs. To address these challenges, we propose DynamoLLM, the first energy-management framework for LLM inference environments. DynamoLLM automatically and dynamically reconfigures the inference cluster to optimize for energy and cost of LLM serving under the service’s performance SLOs. We show that at a service-level, DynamoLLM conserves 53% energy and 38% operational carbon emissions, and reduces 61% cost to the customer, while meeting the latency SLOs.",
    6: "Abstract of paper 6: Large language model (LLM) has recently been considered a promising technique for many fields. This work explores LLM-based wireless network optimization via in-context learning. To showcase the potential of LLM technologies, we consider the base station (BS) power control as a case study, a fundamental but crucial technique that is widely investigated in wireless networks. Different from existing machine learning (ML) methods, our proposed in-context learning algorithm relies on LLM’s inference capabilities. It avoids the complexity of tedious model training and hyper-parameter fine-tuning, which is a well-known bottleneck of many ML algorithms. Specifically, the proposed algorithm first describes the target task via formatted natural language, and then designs the in-context learning framework and demonstration examples. After that, it considers two cases, namely discrete-state and continuous-state problems, and proposes state-based and ranking-based methods to select appropriate examples for these two cases, respectively. Finally, the simulations demonstrate that the proposed algorithm can achieve comparable performance as conventional deep reinforcement learning (DRL) techniques without dedicated model training or fine-tuning. Such an efficient and low-complexity approach has great potential for future wireless network optimization. Index Terms—Large language model, in-context learning, network optimization, transmission power control",
    7: "Abstract of paper 7: Both the training and use of Large Language Models (LLMs) require large amounts of energy. Their increasing popularity, therefore, raises critical concerns regarding the energy efficiency and sustainability of data centers that host them. This paper addresses the challenge of reducing energy consumption in data centers running LLMs. We propose a hybrid data center model that uses a cost-based scheduling framework to dynamically allocate LLM tasks across hardware accelerators that differ in their energy efficiencies and computational capabilities. Specifically, our workload-aware strategy determines whether tasks are processed on energy-efficient processors or high-performance GPUs based on the number of input and output tokens in a query. Our analysis of a representative LLM dataset, finds that this hybrid strategy can reduce CPU+GPU energy consumption by 7.5% compared to a workload-unaware baseline.",
    8: "Abstract of paper 8: Reproducible science requires easy access to data, especially with the rise of data-driven and increasingly complex models used within energy research. Too often however, the data to reconstruct and verify purported solutions in publications is hidden due to some combination of commercial, legal, and sensitivity issues. This early work presents our initial efforts to leverage the recent advancements in Large Language Models (LLMs) to create usable and shareable energy datasets. In particular, we’re utilising their mimicry of human behaviors, with the goal of extracting and exploring synthetic energy data through the simulation of LLM agents capable of interacting with and executing actions in controlled environments. We also analyse and visualise publicly available data in an attempt to create realistic but not quite exact copies of the originals. Our early results show some promise, with outputs that resemble the twin peak curves for household energy consumption. The hope is that our generalised approach can be used to easily replicate usable and realistic copies of otherwise secret or sensitive data.",
    9: "Abstract of paper 9: This paper introduces a method for personalizing energy optimization using large language models (LLMs) combined with an optimization solver. This approach, termed human-guided optimization autoformalism, translates natural language specifications into optimization problems, enabling LLMs to handle various user-specific energy-related tasks. It allows for nuanced understanding and nonlinear reasoning tailored to individual preferences. The research covers common energy sector tasks like electric vehicle charging, HVAC control, and long-term planning for renewable energy installations. This novel strategy represents a significant advancement in context-based optimization using LLMs, facilitating sustainable energy practices customized to individual needs.",
}

#the text for rag is used as an input to the BERT model

#The tokenized inputs are passed to the BERT model for processing.
#(#remember padding=True: Ensures that all inputs are padded to the same length, allowing batch processing.)
#The model outputs a tensor (last_hidden_state), where each input token is represented by a high-dimensional vector.
#last_hidden_state is of shape (batch_size, sequence_length, hidden_size), where:
#batch_size: Number of input texts.
#sequence_length: Length of each tokenized text (after padding).
#hidden_size: Dimensionality of the vector representation for each token (default 768 for bert-base-uncased).

#last_hidden_state[:, 0]: Selects the representation of the [CLS] token for each input text. The [CLS] token is a special token added at the start of each input and is often used as the aggregate representation for the entire sequence.

# -------------------------------------- HINT -----------------------------------
tokenized_inputs = tokenizer(
    list(abstracts_dict.values()),
    return_tensors='pt', # stands for 'PyTorch': tensors will be returned (if omitted, we will get Python lists)
    padding=True,
    truncation=True)   # all the abstracts will be padded to the same length, in order to have a uniformed batch of abstracts => an attention mask will be created
tokenized_inputs = {key: value.to(device) for key, value in tokenized_inputs.items()}   # move the tensors to the device

with torch.no_grad():  # Disable gradient calculation for inference
    abstract_vectors = model_embeddings(**tokenized_inputs).last_hidden_state[:, 0]

#abstract_vectors is a tensor of shape (batch_size, hidden_size) (e.g., (3, 768) in this case), representing each text as a single 768-dimensional vector.

print(abstract_vectors.shape)



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

torch.Size([10, 768])


### Search

With our text data vectorized and indexed, we can now perform searches. We will define a function to search the index for the most relevant documents based on a query.

To perform the search, we need a function (search documents) where we perform the cosine similarity between the query vector and all the abstract vectors. This function will give our the top-k indexes. Once we find the top-k indexes, with another function, we can collect the full text of the documents from the paper dictionary.

To compute cosine similarity, refer to the following formula

```cs = cosine_similarity(vector_a.detach().numpy(), vector_b.detach().numpy())```



In [7]:
import re

from matplotlib.pylab import indices



def get_top_k_similar_indices(query_vector,         # i.e., the QUERY in the RAG system
                              abstract_vectors,     # i.e., the KEY in the RAG system
                              k):

    #Parameters:
    #- query_vector: A tensor of shape (1, hidden_size) representing the query vector.
    #- abstract_vectors: A tensor of shape (batch_size, hidden_size) representing the vector embeddings of the abstract.
    #- k: The number of top indices to return.

    #Returns:
    #- sorted_indices: A numpy array of shape (1, k) containing the indices of the top k most similar abstracts.

    # ------------- HINT ----------------------------------------------------------------
    #Computes the top k indices of the most similar abstracts to the query based on cosine similarity.
    similarities = cosine_similarity(   # from sklearn.metrics.pairwise
        query_vector.cpu().detach().numpy(),    # it is needed to move the tensor on the cpu!
        abstract_vectors.cpu().detach().numpy())

    # IMPORTANT: reason about the size of 'similarities' to understand how to sort it (you can print it)
    # ------------- HINT ----------------------------------------------------------------
    print("Similarities: ", similarities)

    # Sort the indices by similarity in descending order
    sorted_indices = np.argsort(similarities, axis=1)[:, ::-1] # Pay attention to the shape
    print("Sorted indices: ", sorted_indices)

    # Return the top k indices
    return sorted_indices[:, :k]

# NOTE: Real-world RAG systems are implemented with a vectorial database. Semantic search is performed in these databases with optimized algorithms

def remove_bibliography(document_text: str)->str:
     pattern = re.compile(r"\n\s*(?:References|Bibliography|REFERENCES|BIBLIOGRAPHY)\s*\n")
     matches = list(pattern.finditer(document_text))
     if matches:
         last_match = matches[-1]
         return document_text[:last_match.start()].strip()
     return document_text

def retrieve_documents(_indices, documents_dict, max_chars_per_doc=3000):
    """
    Retrieves the documents corresponding to the given indices and concatenates them into a single string.
    :param _indices: A numpy array or list of top-k indices of the most similar documents.
    :param documents_dict: A dictionary where keys are document indices (integers) and values are the document texts (strings).
    :param max_chars_per_doc: Max number of characters per document to retrieve, to avoid OutOfMemoryError when the document is included in a prompt.
    :return: concatenated_documents: A string containing the concatenated texts of the retrieved documents.
    """
    retrieved_docs = [documents_dict[i] for i in _indices[0]]  # Assuming indices is a 2D array
    
    # In order to save space, remove the bibliography from each document
    temp = []
    for d in retrieved_docs:
        temp.append(remove_bibliography(d[:max_chars_per_doc]))
    
    retrieved_docs = temp
    
    # Concatenate the documents into a single string
    _concatenated_documents = " ".join(retrieved_docs)
    
    return _concatenated_documents



# Now, create a vector embedding also for the query

# 1) from NL to tokens

query = "Tell me about Smart EV charging"  # remember to move the tokenised input on the correct device (the same as the model)
tokenized_query = tokenizer(query, return_tensors='pt')
tokenized_query = {key: value.to(device) for key, value in tokenized_query.items()}

# 2) from tokens to vector (i.e., embedding)

query_vector = model_embeddings(**tokenized_query).last_hidden_state[:, 0]
print(f"Model is on device: {model_embeddings.device}")
print(f"Query tensor is on device: {tokenized_query['input_ids'].device}")

_indices = get_top_k_similar_indices(query_vector, abstract_vectors, k=3)
print(f"best 3 indices: {indices}")

concatenated_documents = retrieve_documents(_indices, pdf_texts)

print("Retrieved documents:\n", concatenated_documents)

# TODO: get the top k abstracts similar to the query ( call get_top_k_similar_indices() )

# TODO: get the relative texts ( call retrieve_documents() )





Model is on device: cuda:0
Query tensor is on device: cuda:0
Similarities:  [[0.6623689  0.45562398 0.5043023  0.369049   0.3667243  0.3431152
  0.5163578  0.4323544  0.41912878 0.5159153 ]]
Sorted indices:  [[0 6 9 2 1 7 8 3 4 5]]
best 3 indices: <function indices at 0x7e13347cb740>
Retrieved documents:
 XIWU: A BASIS FLEXIBLE AND LEARNABLE LLM FOR HIGH
ENERGY PHYSICS
Zhengde Zhang1, Yiyu Zhang1, Haodong Yao1, Jianwen Luo2, Rui Zhao1, Bo Huang1, Jiameng Zhao3,1,
Yipu Liao1, Ke Li1, Lina Zhao1, Jun Cao1, Fazhi Qi1,∗, and Changzheng Yuan1,†
1Institute of High Energy Physics, Chinese Academy of Sciences, Beijing 100049, China
2School of Physical Sciences, University of Chinese Academy of Sciences, Beijing 100049, China
3School of Computer and Artificial Intelligence, ZhengZhou University, Henan 450066, China
ABSTRACT
Large Language Models (LLMs) are undergoing a period of rapid updates and changes, with state-
of-the-art (SOTA) model frequently being replaced. When applying LLMs to a spe

### A function to perform Retrieval Augmented Generation

In this step, we’ll combine the context retrieved from our documents with LLAMA to generate responses. The context will provide the necessary information to the model to produce more accurate and relevant answers.

In [8]:

query_for_rag = "What are the main energy issues for Large Language Models?"

new_system_prompt = """You are a helpful, respectful and honest assistant. \
    Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, \
    unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses \
    are socially unbiased and positive in nature. If a question does not make any sense, or is not factually \
    coherent, explain why instead of answering something not correct. If you don't know the answer to a question, \
    please don't share false information.
    Provide straight to the point answers, without bullet points."""


#now embed the result of the previous prompt in a new prompt to ask some additional information (i.e., the year of foundation). you have to split the original prompt to create a new one

p_not_augmented =  query_for_rag

# We want to query Llama, and not to compute the embeddings
text_generation_pipeline = load_model("meta-llama/Llama-3.2-3B-Instruct", quantization=True)  # We use quantization in order to save memory to be able to manage larger prompts
messages = [
    {'role': 'system', 'content': new_system_prompt},
    {'role': 'user', 'content': p_not_augmented}
]


print("PROMPT:\n\n")
print(p_not_augmented)

print("\n\n\n")

print("RESPONSE:\n\n")
print(text_generation_pipeline(messages, max_new_tokens = 200)[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:


What are the main energy issues for Large Language Models?




RESPONSE:


Large Language Models (LLMs) face several energy-related challenges. One of the primary concerns is the massive computational power required to train and run these models. This is due to the complex neural network architecture and the large number of parameters involved.

Another significant issue is the high energy consumption of data centers and servers that host LLMs. These data centers often rely on non-renewable energy sources, which contributes to greenhouse gas emissions and climate change.

Additionally, the training process itself is energy-intensive, as it involves matrix multiplications and other computationally expensive operations. This can lead to significant energy consumption, especially for large-scale models.

Finally, there is also a concern about the e-waste generated by the rapid obsolescence of LLM hardware and the need for frequent hardware upgrades, which can contribute to elect

In [9]:
# Now the same for the augmented prompt


p_augmented =  """{PROMPT} \n\nCollect information from the following article: {BODY}""".format(BODY=concatenated_documents, PROMPT=query_for_rag)

messages = [
    {'role': 'system', 'content': new_system_prompt},
    {'role': 'user', 'content': p_augmented}
]


print("PROMPT:\n\n")
print(p_augmented)

print("\n\n\n")

print("RESPONSE:\n\n")
response = text_generation_pipeline(messages, max_new_tokens = 200,
                                    truncation=True)    # Truncation has been set to avoid OutOfMemoryError in case of long prompts (ATTENTION: only the first part of the prompt will be provided to the model, which is still not optimal)
response = response[0]["generated_text"][-1]["content"] # Don't forget this part
print(response)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:


What are the main energy issues for Large Language Models? 

Collect information from the following article: XIWU: A BASIS FLEXIBLE AND LEARNABLE LLM FOR HIGH
ENERGY PHYSICS
Zhengde Zhang1, Yiyu Zhang1, Haodong Yao1, Jianwen Luo2, Rui Zhao1, Bo Huang1, Jiameng Zhao3,1,
Yipu Liao1, Ke Li1, Lina Zhao1, Jun Cao1, Fazhi Qi1,∗, and Changzheng Yuan1,†
1Institute of High Energy Physics, Chinese Academy of Sciences, Beijing 100049, China
2School of Physical Sciences, University of Chinese Academy of Sciences, Beijing 100049, China
3School of Computer and Artificial Intelligence, ZhengZhou University, Henan 450066, China
ABSTRACT
Large Language Models (LLMs) are undergoing a period of rapid updates and changes, with state-
of-the-art (SOTA) model frequently being replaced. When applying LLMs to a specific scientific
field, it’s challenging to acquire unique domain knowledge while keeping the model itself advanced.
To address this challenge, a sophisticated large language model system 